In [1]:
# conda env 'at3d'

import sys
sys.path.append('/data/keeling/a/johntl3/b/LundstromPub2025/src')

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import at3d
import MISR_preprocess2 as pp
import util

In [2]:
misr_data_file = '../data/observations/CAMP2Ex_RF17_MISR_OBS_000.nc'
rte_grid_outfile = '../data/rte_grids/rte_grid.nc'
obs_outfile = '../data/observations/obs.nc'

In [3]:
# Create radiative transfer grid (for the bounding box)
z = np.concatenate((np.array([0, 0.2, 0.39]), np.arange(0.4, 1.5, 0.1), np.array([1.41, 1.6, 1.8])))
rte_grid = at3d.grid.make_grid(0.275, 65, 0.275, 65, z)
rte_grid.to_netcdf(rte_grid_outfile)

# Read in MISR data in Local North coordinates
misr_data_LN = xr.load_dataset(misr_data_file)

# Convert MISR data to ENU coordinates
misr_data_ENU = pp.LN2ENU(misr_data_LN)

# Horizontal shift to place origin at the corner of the rte grid (maintain centering)
misr_data_ENU['E'] = misr_data_ENU.E + np.max(rte_grid.x.data)/2
misr_data_ENU['E'] = misr_data_ENU.E.assign_attrs(units='km')
misr_data_ENU['N'] = misr_data_ENU.N + np.max(rte_grid.y.data)/2
misr_data_ENU['N'] = misr_data_ENU.N.assign_attrs(units='km')

# Project rays to the top of the domain
U2 = np.ones(misr_data_ENU.mu.data.shape)*np.max(rte_grid.z.data)
misr_data_ENU = pp.project_to_U2(misr_data_ENU, U2)

# Wind correction - CMV comes from the MISR CMV product - 
# see ./ancillary/CAMP2Ex_RF17_MISR_CMV.ipynb
s = 1.59 # m/s
theta = 35 # degrees
u = s * np.sin(np.radians(theta)) # m/s
v = s * np.cos(np.radians(theta)) # m/s
misr_data_ENU = misr_data_ENU.assign_attrs(u=u, v=v)
misr_data_ENU['dt'] = (['camera'], -1*(np.array([0, 60, 113, 159, 204, 249, 295, 348, 408])-204))
misr_data_ENU['dt'] = misr_data_ENU.dt.assign_attrs(units='s')
misr_data_ENU['dt'] = misr_data_ENU.dt.assign_attrs(description='Time shift needed to register at AN camera')
misr_data_ENU['dE'] = misr_data_ENU.dt * u * 1e-3
misr_data_ENU['dE'] = misr_data_ENU.dE.assign_attrs(units='km')
misr_data_ENU['dN'] = misr_data_ENU.dt * v * 1e-3
misr_data_ENU['dN'] = misr_data_ENU.dN.assign_attrs(units='km')
misr_data_ENU['E2'] = misr_data_ENU.E2 + misr_data_ENU.dE
misr_data_ENU['N2'] = misr_data_ENU.N2 + misr_data_ENU.dN

# Calculate the maximum extent at oblique angles and pad
pad = 2 # km
min_extent, max_extent = pp.TOA_extent(rte_grid, misr_data_ENU)
min_extent = min_extent - pad
max_extent = max_extent + pad

# Store observations in a sensor dict
obs = at3d.containers.SensorsDict()
wavelength = 0.67
for cam in misr_data_ENU.camera.data:
    obs.add_sensor('MISR', pp.make_sensor(misr_data_ENU, cam, min_extent, max_extent, wavelength))

# Save obs
util.save_measurements(obs_outfile, obs)